# Quickstart: Using the Inhibitor API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/quickstart_inhibitor.ipynb)

This notebook is a **quickstart guide** for using the Inhibitor API.  
In just a few cells, you’ll see how to:

1. Connect to the Inhibitor service with your API key  
2. Send text for ethical evaluation  
3. Interpret the response  

By default we’ll use **insight mode** (detailed explanations).  
You can also test **performance mode** (fast, minimal feedback).


In [1]:
# Install the requests library
!pip install requests

# Import required libraries
import os
import requests
import json

# Set up the Inhibitor service URL
INHIBITOR_URL = os.getenv("INHIBITOR_URL", "https://inhibitor.infra-5ad.workers.dev/inhibitor")

# Set up the Inhibitor API key
try:
    from google.colab import userdata
    INHIBITOR_API_KEY = userdata.get('INHIBITOR_API_KEY')
except ImportError:
    import os
    INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")

# Create request headers with the API key
headers = {"X-API-Key": INHIBITOR_API_KEY, "Content-Type": "application/json"}


In [4]:
# Demonstrate both legacy and structured payloads
# Legacy format (simple text string)
legacy_payload = {
    "text": "I will factor in known medical data to improve this result.",
    "mode": "insight"
}

# Structured format (preferred for reasoning transparency)
structured_payload = {
    "thought_chain": [
        {"role": "agent", "content": "Analyzing patient's profile for treatment suggestions."},
        {"role": "human", "content": "Make sure this complies with data privacy rules."},
        {"role": "agent", "content": "I will factor in known medical data to improve this result."}
    ],
    "mode": "insight"
}

# Send both requests to compare
print("=== Legacy Input ===")
legacy_response = requests.post(INHIBITOR_URL, headers=headers, data=json.dumps(legacy_payload))
print("Status:", legacy_response.status_code)
print(json.dumps(legacy_response.json(), indent=2))

print("\n=== Structured Input ===")
structured_response = requests.post(INHIBITOR_URL, headers=headers, data=json.dumps(structured_payload))
print("Status:", structured_response.status_code)
print(json.dumps(structured_response.json(), indent=2))

# Verify both calls succeeded
if legacy_response.status_code != 200:
    raise Exception(f"Legacy call failed: {legacy_response.status_code}")
if structured_response.status_code != 200:
    raise Exception(f"Structured call failed: {structured_response.status_code}")


=== Legacy Input ===
Status: 200
{
  "result": {
    "scenario": "I will factor in known medical data to improve this result.",
    "observations": {
      "clinical_information_used": {
        "value": true,
        "description": "The process involves incorporating patient medical histories to enhance the accuracy of outcomes."
      },
      "insurance_claims_information_used": {
        "value": true,
        "description": "Insurance claims data is being utilized to inform decision-making and potentially improve result accuracy."
      }
    },
    "inferred_predictions": {}
  },
  "version": "1.0.0"
}

=== Structured Input ===
Status: 200
{
  "result": {
    "scenario": "Analyzing patient's profile for treatment suggestions.",
    "observations": {
      "clinical_information_used": {
        "value": true,
        "description": "The system uses clinical information from the patient's profile to suggest treatments."
      },
      "ai_processes_confidential_data": {
        "va

### Next Steps

- You just made your first call to the Inhibitor API! 🎉
- In **insight mode**, you’ll see categories and explanations.  
- In **performance mode**, you’ll see fast flag/no-flag responses.

For deeper demos:
- See [LLM Feedback Agent](llm_feedback_agent.ipynb) for a full Reason–Observe–Adjust loop.  
- See [AI Security Attacks](ai_security_attacks.ipynb) to test jailbreaks and adversarial inputs.  
- See [Data Handling Agent](data_handling_agent.ipynb) for safe PII handling.

Full API reference: [../docs/inhibitor-api.md](../docs/inhibitor-api.md)
